Run with the installed Brain-Score packages. Obtain authorized dataset access before running; these notebooks do not include the datasets.


# Pereira2018 Ridge Ceiling Computation

Pre-computes ceilings for the `Pereira2018.{243,384}sentences-ridge` benchmarks
and saves them as local `.nc` files in `ceilings/`.

The ridge benchmarks currently lack pre-computed ceilings, so the ceiling is
recomputed from scratch on every `score()` call. After running this notebook,
the benchmark loads from the local `.nc` files instead.

In [ ]:
import logging
import pickle
import sys
import warnings
from pathlib import Path

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
for shush_logger in ['botocore', 'boto3', 's3transfer', 'urllib3']:
    logging.getLogger(shush_logger).setLevel(logging.INFO)
warnings.filterwarnings('ignore')

In [ ]:
from brainscore_language import load_dataset, load_metric
from brainscore_language.benchmarks.pereira2018.ceiling_packaging import ExtrapolationCeiling
from brainscore_core.supported_data_standards.brainio.packaging import write_netcdf

import brainscore_language.benchmarks.pereira2018 as _pereira_pkg
# Write results outside the installed package.
CEILING_DIR = Path.cwd() / 'pereira-ceilings'
CEILING_DIR.mkdir(exist_ok=True)
print(f'Saving ceilings to: {CEILING_DIR}')

In [ ]:
def load_data(experiment: str, atlas: str = 'language'):
    """Load Pereira2018 data, matching _Pereira2018Experiment._load_data."""
    lang_data = load_dataset('Pereira2018.language')
    data = load_dataset(f'Pereira2018.{atlas}')
    data.coords['presentation'] = lang_data.coords['presentation']
    data = data.sel(experiment=experiment)
    data = data.dropna('neuroid')
    # Use a ridge-specific identifier so the @store cache in
    # ExtrapolationCeiling.collect does not collide with linear ceilings.
    data.attrs['identifier'] = f'Pereira2018.language.{experiment}.ridge'
    return data

In [ ]:
def compute_ceiling(experiment: str):
    """Compute the ridge ceiling, with pickle caching to survive notebook restarts."""
    cache_path = CEILING_DIR / f'ceiling_{experiment}_ridge.pkl'
    if cache_path.exists():
        print(f'Loading cached ceiling from {cache_path}')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    data = load_data(experiment)
    metric = load_metric('ridge_pearsonr', crossvalidation_kwargs=dict(
        split_coord='story',
        kfold='group',
        random_state=1234,
    ))

    ceiler = ExtrapolationCeiling()
    ceiling = ceiler(assembly=data, metric=metric)
    print(f'{experiment} ceiling: {float(ceiling):.4f}')

    # Cache the ceiling object so we don't redo the 68-min extrapolation
    with open(cache_path, 'wb') as f:
        pickle.dump(ceiling, f)
    print(f'Cached ceiling to {cache_path}')

    return ceiling

In [ ]:
def save_ceiling_nc(ceiling, experiment: str):
    """Save all three nesting levels as .nc files for the benchmark to load."""
    identifier = f'Pereira2018.{experiment}-ridge'

    for prefix, assembly in [
        ('ceiling_', ceiling),
        ('ceiling_raw_', ceiling.raw),
        ('ceiling_raw_raw_', ceiling.raw.raw),
    ]:
        filename = f"{prefix}{identifier.replace('.', '_')}.nc"
        path = CEILING_DIR / filename
        write_netcdf(assembly, path)
        print(f'  Saved {path.name} ({path.stat().st_size / 1e6:.1f} MB)')

## 384sentences

In [ ]:
ceiling_384 = compute_ceiling('384sentences')
save_ceiling_nc(ceiling_384, '384sentences')

## 243sentences

In [ ]:
ceiling_243 = compute_ceiling('243sentences')
save_ceiling_nc(ceiling_243, '243sentences')

In [ ]:
from brainscore_language import score

model = 'oasm-sigma0.6'
benchmark = 'Pereira2018.384sentences-ridge'

In [ ]:
score = score(model, benchmark)

In [ ]:
score